# THERMA SEARCH AI · 06 · HIT-UAV Error Analysis & Controlled Experiment Preparation

**Çalışma zamanı → Tümünü çalıştır.** GPU önerilir; CPU çıkarımı da desteklenir. İlk Google Drive izni gerekebilir. **Bu notebook eğitim başlatmaz.** Mevcut HIT-UAV checkpoint’i yalnızca özgün **validation** splitinin önceki politika ile filtrelenmiş kopyasında incelenir; train/test görüntüleri açılmaz ve split değiştirilmez.

Sıfır ham anotasyonlu ve DontCare içeren görüntüler Phase 04 ile aynı şekilde dışlanır; diğer-sınıf-only görüntüler önceki Person-arka plan politikasıyla korunur. Eski modeller, raporlar ve kaynaklar değiştirilmez.

WiSARD sonuçları zaten incelenmiş tanılama bağlamıdır: 261 görüntü/1006 GT, 207 TP/532 FP/799 FN; mAP50=.1453, mAP50:95=.0415. **Bunlar bu notebook’un yeni skorları değildir.** Sağlanan ön analiz paketinde 261 satırlık CSV/özet aritmetiği incelenmiştir; ham tahminler yoktur. 410 düşük güvenli native örtüşme adayı bire bir eşlenmemiştir ve kurtarılabilir TP değildir. WiSARD hiçbir threshold, hiperparametre veya checkpoint seçiminde kullanılmaz; pristine final holdout sayılmaz.

## 1 · Sınanan kod ve mevcut çalışma zamanı
Ultralytics 8.3.228 pin’i korunur. Paket/kod kurulumu model veya veri indirmesi değildir. Checkpoint ve kaynak ZIP önceden mevcut olmalıdır.

In [ ]:
from pathlib import Path
import os
import sys
import subprocess
import json

REPOSITORY='https://github.com/hilalaltunayy/therma-search-ai.git'
CODE_REF='5dfaab3a38d273346887a5ab6ce23f6318484d6e'  # Native kategori düzeltmesi ve preflight içeren sınanmış commit.
local_code_root=os.environ.get('THERMA_CODE_ROOT')
if local_code_root:
    PROJECT_ROOT=Path(local_code_root).resolve()
else:
    PROJECT_ROOT=Path('/content/therma-search-ai-phase06')
    if not PROJECT_ROOT.exists():
        subprocess.run(['git','clone','--quiet',REPOSITORY,str(PROJECT_ROOT)],check=True)
    if subprocess.check_output(['git','-C',str(PROJECT_ROOT),'remote','get-url','origin'],text=True).strip()!=REPOSITORY:
        raise RuntimeError('Beklenmeyen kod deposu.')
    if subprocess.check_output(['git','-C',str(PROJECT_ROOT),'status','--porcelain'],text=True).strip():
        raise RuntimeError('Yerel kod değişikliği var; üzerine yazılmadı. Temiz Colab çalışma zamanı kullanın.')
    subprocess.run(['git','-C',str(PROJECT_ROOT),'fetch','--quiet','--depth','1','origin',CODE_REF],check=True)
    subprocess.run(['git','-C',str(PROJECT_ROOT),'checkout','--quiet','--detach','FETCH_HEAD'],check=True)
    subprocess.run([sys.executable,'-m','pip','install','--quiet','--upgrade-strategy','only-if-needed','-r',str(PROJECT_ROOT/'requirements-training-colab.txt')],check=True)
sys.path.insert(0,str(PROJECT_ROOT))
for module in list(sys.modules):
    if module=='src' or module.startswith('src.'):
        del sys.modules[module]
from src.evaluation.phase06 import run_phase06
from IPython.display import display, Image as DisplayImage
GIT_COMMIT=subprocess.check_output(['git','-C',str(PROJECT_ROOT),'rev-parse','HEAD'],text=True).strip()
print('Kod revizyonu:',GIT_COMMIT)

## 2 · Drive ve mevcut girdiler
Varsayılan checkpoint, önceki HIT-UAV baseline run’ındandır. Aynı klasördeki eğitim veri manifesti mevcutsa validation üyeliği/hash’leri karşılaştırılır; yoksa bu sınır açıkça raporlanır. WiSARD ZIP’i varsa **yalnızca piksel/GT boyut/kontrast betimlemesi** yapılır, WiSARD model çıkarımı yapılmaz. ZIP yoksa bu karşılaştırma ölçülmüş gibi sunulmaz; kaynak validation analizi devam eder.

Önceki native export varsa aşağıdaki hücre modeli yüklemeden salt-okunur preflight yapar. **Yalnız parser kontrolü için ilk iki kod hücresini çalıştırıp burada durun.** Tümünü çalıştır sonraki hücrede yeni değerlendirme başlatır.

In [ ]:
if not local_code_root:
    from google.colab import drive
    drive.mount('/content/drive')
CHECKPOINT=Path(os.environ.get('THERMA_PHASE06_MODEL','/content/drive/MyDrive/ThermaSearchAI/runs/hit_uav_yolo11s/run_yolo11s_20261120T151540Z_78fb2af9/best.pt'))
HIT_UAV_ZIP=Path(os.environ.get('THERMA_PHASE06_HIT_ZIP','/content/drive/MyDrive/ThermaSearchAI/datasets/hit_uav_v1.2.1.zip'))
WISARD_ZIP=Path(os.environ.get('THERMA_PHASE06_WISARD_ZIP','/content/drive/MyDrive/ThermaSearchAI/datasets/WiSARD_Multi_Modal_Sample.zip'))
OUTPUT_ROOT=Path(os.environ.get('THERMA_PHASE06_OUTPUT','/content/drive/MyDrive/ThermaSearchAI/reports/phase06_error_analysis'))
WORK_ROOT=Path(os.environ.get('THERMA_PHASE06_WORK','/content/therma-phase06-work'))
print('Checkpoint:',CHECKPOINT.is_file(),CHECKPOINT)
print('HIT-UAV ZIP:',HIT_UAV_ZIP.is_file(),HIT_UAV_ZIP)
print('WiSARD betimsel ZIP:',WISARD_ZIP.is_file())
print('Yeni rapor dizinleri:',OUTPUT_ROOT)
from src.evaluation.native_preflight import latest_native_run, preflight_native_export
PREVIOUS_NATIVE_RUN=latest_native_run(OUTPUT_ROOT,required=False)
if PREVIOUS_NATIVE_RUN is None:
    print('Preflight: önceki native JSON + veri manifesti bulunamadı; yeni analiz devam edebilir.')
else:
    print('Salt-okunur native parser preflight:')
    print(json.dumps(preflight_native_export(PREVIOUS_NATIVE_RUN),ensure_ascii=False,indent=2))


## 3 · Kaynak validation tanılaması
Sabit conf=.25, eşleme IoU=.50, NMS IoU=.70, max_det=300, imgsz=640 kullanılır. Native validation conf tabanı=.001 ile ayrı çalışır; native mAP ile özel TP/FP/FN karıştırılmaz.

FN/FP maksimum-IoU grupları, olası yinelenen tahminler, küçük nesne recall’u ve confidence dağılımları kaydedilir. Native düşük güvenli aday aynı anda birden fazla GT ile örtüşebilir: kurtarılabilir TP sayılmaz. Native batch/padding ve fixed predictor etkin ayarları izlenir. GT/prediction görsellerinde kırmızı GT, mavi tahmin, mor FP ve turuncu kesik FN’dir.

Yoğunluk ve kontrast, 8-bit display luminance üzerinden tanımlanır; sıcaklık veya radyometrik kalibrasyon ölçümü değildir. GT ROI/çevre farkları başka nesneler ve sahne içeriğinden etkilenebilir. Karşılaştırmalar betimseldir; neden iddiası kurulmaz.

In [ ]:
PHASE06_RUN_DIR, PHASE06_SUMMARY=run_phase06(
    CHECKPOINT,HIT_UAV_ZIP,OUTPUT_ROOT,WORK_ROOT,
    git_commit=GIT_COMMIT,wisard_zip=WISARD_ZIP,project_root=PROJECT_ROOT,
)

## 4 · Yeni kaynak ölçümleri ve kanıt dosyaları
Sonuçlar yalnızca tamamlanan kaynak validation çıkarımına aittir. Validation eski checkpoint seçiminde de kullanılmıştır; bağımsız final test değildir. Eksik/bozuk girdide notebook açık hata ile durur ve yeni run’da tanılama raporu kalır.

In [ ]:
print('Durum:',PHASE06_SUMMARY['status'])
print('Yeni HIT-UAV val hata sayımları:',json.dumps(PHASE06_SUMMARY['source_validation_diagnosis'],ensure_ascii=False,indent=2))
print('Native kaynak-val mAP (ayrı prosedür):',PHASE06_SUMMARY['source_native_map'])
print('WiSARD piksel karşılaştırması:',PHASE06_SUMMARY['wisard_descriptive_status'])
for pattern in ['confidence_diagnostics.png','descriptive_domain_comparison.png','missing_detection_01.png',
                'poor_localization_01.png','duplicate_candidate_01.png','small_object_01.png']:
    for path in sorted((PHASE06_RUN_DIR/'figures').glob(pattern)):
        display(DisplayImage(filename=str(path),width=1100))
print('Kaynaklar değişmedi:',PHASE06_SUMMARY['inputs_unchanged'])
print('Rapor:',PHASE06_RUN_DIR/'error_analysis_report.md')
print('Faz 07 önerisi:',PHASE06_RUN_DIR/'phase07_experiment_matrix.md')
print('Eğitim başlatılmadı; WiSARD model/threshold seçimine kapalıdır.')

## 5 · Faz 07 önerisi — çalıştırılacak eğitim hücresi yok
Yeni ortak kaynak train/val kontrolü üzerinden ayrı kollar: **imgsz=960**, **scale=0.25** ve **mosaic=0.0**. Her kolda tek ana faktör değişir; bu kollar birleştirilmez. Gerçek baseline etkin optimizer/lr/augmentasyon ayarları doğrulanmadan öneriler yürütülemez.

960 bellek maliyeti nedeniyle önerilen batch=4 henüz ölçülmüş uygunluk sonucu değildir. Batch/optimizer/FP32/effective batch bütün yeni kollar için sabitlenir; gerekirse kontrol de yeniden çalıştırılır. Eski batch=8 run ile doğrudan tek-faktör kazanım iddiası kurulmaz. Kaynak validation ile önceden tanımlı seçim, eşlenmiş seed tekrarları ve tüm başarısız run’ların kaydı gerekir.

WiSARD zaten görülmüştür. Final genelleme için ayrı uçuş/saha/görevlerden bağımsız dış veri ve doğrulanmış negatif sahneler önerilir; yeni holdout model/protokol dondurulana kadar kapalı tutulur. Plan **onaylanmamış ve çalıştırılmamış öneridir**; bu notebook eğitim başlatmaz.